In [1]:
import numpy as np
import pandas as pd
import psycopg2
import datetime

In [2]:
## add proper credentials

conn = psycopg2.connect("dbname=kkdj20_test user=sebastian host=localhost password=123456")
#conn = psycopg2.connect("dbname=kraladies user=sebastian host=localhost password=123456")

cur = conn.cursor()

In [5]:
# use one or the other
# EVENT_NAME='KKM11'
EVENT_ID=9

verbose = False

try:
    # select event_id with of <EVENT_NAME>
    cur.execute("SELECT id FROM baskets_event AS be WHERE be.event_name LIKE %s", (EVENT_NAME,))
    res = cur.fetchall()
    if res:
        event_id = int(res[0][0])
except:
    event_id = EVENT_ID


cur.execute("SELECT * FROM baskets_event as be WHERE be.id = %s", (EVENT_ID,))
rn=cur.fetchone()
    
total_sum=0
#if res:
#    event_id = int(res[0][0])
    # select all baskets in <EVENT_NAME>
cur.execute("SELECT id FROM baskets_basket as bb WHERE bb.event_id=%s",(event_id,))
res = cur.fetchall()
print (f"There are a total of {len(res)} baskets assigned to event_id={event_id}\n")

nonempty = 0
nitems = 0
for bid in res:
    basket_id = bid[0]
    if verbose:
        print ("Basket No{}".format(basket_id))
    cur.execute('''SELECT bi.id
                        , bi.price
                        , bi."vendorID_id" AS vid
                        , bv.vendor_number as vnr
                        , bi.created
                    FROM baskets_item AS bi 
                    JOIN baskets_vendor as bv 
                        ON bv.id=bi."vendorID_id"
                    WHERE bi.basket_id=%s
                ''',(basket_id,))

    r2 = cur.fetchall()
    _sum=0
    for iid in r2:
        item_id=iid[0]
        price=iid[1]
        if price>0:
            nitems+=1
        vendor_id=iid[2]
        vendor_nr=iid[3]
        item_created=iid[4]
        _sum+=price
        total_sum +=price
        #print ("  Item{}: {:.2f} Euro  (Vendor #{})".format(item_id, price, vendor_nr))
        if verbose:
            print ("  Item{}: {:.2f} Euro  (Vendor #{}) ({})".format(item_id, price, vendor_nr, item_created))
    if verbose:
        print ("   ...Total = {:.2f} Euro".format(_sum))
    if _sum  > 0:
        nonempty +=1

dashes = 55*"-"
print (dashes)
#print ("Gesamt Total: {:.2f} Euro\n\tverteilt auf {:d} nicht-leere Warenkoerbe (= {:.2f} Euro/Korb).".format(total_sum, nonempty, total_sum/nonempty))
print (f"EVENT_ID {rn[0]}: {rn[1]} -- {rn[3]}")
print (dashes)
print ("Gesamt Total: \t{:.2f} Euro\nAnzahl Koerbe: \t{:d} \t\t(= {:.2f} Euro/Korb)".format(total_sum, nonempty, total_sum/nonempty))
print ("Anzahl Teile: \t{:d} \t\t(= {:.2f} Euro/Teil)".format(nitems, total_sum/nitems))
print (dashes)            

There are a total of 397 baskets assigned to event_id=9

-------------------------------------------------------
EVENT_ID 9: KKM15 -- 15. KKM
-------------------------------------------------------
Gesamt Total: 	11617.35 Euro
Anzahl Koerbe: 	361 		(= 32.18 Euro/Korb)
Anzahl Teile: 	4275 		(= 2.72 Euro/Teil)
-------------------------------------------------------


In [4]:
## More SQL-ish way to get number of items of one event
event_id = 8
cur.execute('''SELECT COUNT(bi.id) 
            FROM baskets_item as bi 
            JOIN baskets_basket as bb ON bb.id=bi.basket_id
            WHERE bb.event_id=%s''',(event_id,))
res = cur.fetchone()
#print (res)
print (f"There are a total of {res[0]} items assigned to event_id={event_id}\n")

There are a total of 5153 items assigned to event_id=8

